# 05 — Raising Exceptions

## Exception Handling

The previous notebooks taught how to **handle exceptions that occur**. This notebook introduces the other side:

> **How to deliberately raise an exception when a condition in your program is invalid.**

The main keyword is:

```python
raise
```

### Learning goals

By the end of this notebook, you should understand:

- what `raise` does
- how to raise built-in exceptions
- how to raise exceptions conditionally
- how `raise` works with user input
- how functions can raise exceptions
- raising vs returning error-like values
- how to capture and re-raise exceptions
- how to write useful exception messages
- how `raise` can be used in data validation and OOP
- the difference between `TypeError` and `ValueError`
- how callers can handle exceptions raised by functions

> **Boundary:** Custom exception classes are reserved for the next notebook, `06_Custom_Exceptions.ipynb`.

## 1. Introduction

The previous notebooks followed this pattern:

```text
Exception occurs
      ↓
We handle it
```

This notebook introduces another situation:

```text
We detect an invalid condition
      ↓
We raise an exception
      ↓
The exception can be handled
```

For example:

In [ ]:
age = -5

if age < 0:
    raise ValueError("Age cannot be negative")

Python can store `-5` as an integer, but an application may decide that a negative age is invalid.

The program can explicitly enforce that rule by raising an exception.

## 2. What Is `raise`?

`raise` is used to explicitly trigger an exception.

Basic syntax:

```python
raise ExceptionType("message")
```

Example:

In [ ]:
raise ValueError("Invalid value")

When this statement executes, Python raises the specified exception.

If nothing catches it, the exception propagates and normal execution stops at that point.

## 3. Basic `raise` Example

Consider:

In [ ]:
age = -10

if age < 0:
    raise ValueError("Age cannot be negative")

The execution steps are:

1. `age` is checked.
2. The condition is true.
3. `ValueError` is raised.
4. If the exception is not handled, the program stops at that point.

The important idea is that **your program decides when a condition is invalid**.

## 4. Why Do We Need `raise`?

Python does not automatically know every business rule of your program.

For example:

```python
age = -5
```

Python knows this is an integer.

But your application may decide:

> An age cannot be negative.

You can enforce that rule:

```python
if age < 0:
    raise ValueError("Age cannot be negative")
```

This is especially useful for input validation and enforcing program rules.

## 5. Raising Common Built-in Exceptions

You can raise many built-in exception types.

### `ValueError`

```python
raise ValueError("Invalid value")
```

### `TypeError`

```python
raise TypeError("Expected a number")
```

### `ZeroDivisionError`

```python
raise ZeroDivisionError("Division by zero is not allowed")
```

### `IndexError`

```python
raise IndexError("Invalid index")
```

Choose an exception type that appropriately describes the problem.

In [ ]:
raise ValueError("Invalid value")

The example above intentionally raises an exception. In later examples, exceptions will usually be raised conditionally or caught so the complete flow can be observed.

## 6. `raise` with Conditions

One of the most important patterns is raising an exception when a validation condition fails.

```python
marks = 120

if marks < 0 or marks > 100:
    raise ValueError("Marks must be between 0 and 100")
```

If:

```python
marks = 85
```

nothing is raised.

If:

```python
marks = 120
```

a `ValueError` occurs.

In [ ]:
def validate_marks(marks):
    if marks < 0 or marks > 100:
        raise ValueError("Marks must be between 0 and 100")
    return marks


print(validate_marks(85))

The valid value passes the validation.

An invalid value would cause the function to raise `ValueError`.

## 7. `raise` with User Input

`raise` can be combined with input validation.

In [ ]:
try:
    age = int(input("Enter your age: "))

    if age < 0:
        raise ValueError("Age cannot be negative")

    print("Valid age:", age)

except ValueError as e:
    print("Error:", e)

This combines:

- user input
- conversion
- validation
- exception raising
- exception handling

The important distinction is that `int()` may raise `ValueError` during conversion, while the validation condition can also explicitly raise `ValueError`.

## 8. `raise` with `try-except`

Raising and handling can work together.

```python
try:
    age = int(input("Enter age: "))

    if age < 0:
        raise ValueError("Age cannot be negative")

except ValueError as e:
    print("Error:", e)
```

The flow is:

```text
Validation
    ↓
raise
    ↓
exception
    ↓
except
    ↓
handle
```

The exception does not have to originate automatically from Python's built-in operations. Your own validation code can raise it.

In [ ]:
try:
    age = int(input("Enter age: "))

    if age < 0:
        raise ValueError("Age cannot be negative")

except ValueError as e:
    print("Error:", e)

## 9. Raising an Exception Inside a Function

Functions can raise exceptions when their input violates a required condition.

In [ ]:
def calculate_square_root(number):
    if number < 0:
        raise ValueError("Number cannot be negative")

    return number ** 0.5


print(calculate_square_root(25))

For:

```python
calculate_square_root(-4)
```

the function raises `ValueError`.

The function does not silently produce a meaningless result for an input that violates its rule.

## 10. Why Functions Should Raise Exceptions

A function should not always silently return an incorrect or meaningless value.

For example:

```python
def divide(a, b):
    if b == 0:
        return None
```

The caller must remember that `None` means something went wrong.

Instead, the function can explicitly communicate the invalid condition:

```python
def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("Cannot divide by zero")

    return a / b
```

Then the caller decides how to handle the exception.

In [ ]:
def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("Cannot divide by zero")

    return a / b


try:
    print(divide(10, 0))
except ZeroDivisionError as e:
    print("Error:", e)

The function detects the invalid condition, while the caller controls the response.

## 11. Raising vs Returning an Error

Compare the two approaches.

### Returning an error-like value

```python
def withdraw(amount):
    if amount < 0:
        return "Invalid amount"
```

### Raising an exception

```python
def withdraw(amount):
    if amount < 0:
        raise ValueError("Amount cannot be negative")
```

An exception clearly signals:

> This operation could not be completed normally.

The caller can then decide how to handle it.

A returned string is simply a normal return value and can be accidentally treated like successful output.

## 12. `raise` Without an Exception Object

There is an important difference between:

```python
raise
```

and:

```python
raise ValueError("message")
```

A bare:

```python
raise
```

is primarily used **inside an exception handler to re-raise the currently handled exception**.

Example:

In [ ]:
try:
    number = int("abc")
except ValueError:
    print("Logging the error")
    raise

The handler performs a local action, such as logging, and then `raise` propagates the same exception again.

A bare `raise` is not used as a general replacement for:

```python
raise ValueError(...)
```

## 13. Re-Raising an Exception

Consider a function that performs some local processing:

In [ ]:
def process_data():
    try:
        value = int("abc")
    except ValueError:
        print("Error occurred inside function")
        raise


try:
    process_data()
except ValueError:
    print("Handled by caller")

The flow is:

```text
process_data()
      ↓
exception occurs
      ↓
inner except
      ↓
log/process
      ↓
raise
      ↓
caller receives exception
```

This is useful when a lower-level function needs to perform some local action but should not hide the exception from its caller.

## 14. `raise` and Exception Messages

Good exception messages make debugging easier.

Instead of:

```python
raise ValueError("Error")
```

prefer a descriptive message:

```python
raise ValueError("Age must be greater than or equal to 0")
```

Another example:

```python
raise ValueError("Marks must be between 0 and 100")
```

The message should help identify what was invalid.

In [ ]:
def validate_age(age):
    if age < 0:
        raise ValueError("Age must be greater than or equal to 0")

    return age


try:
    validate_age(-5)
except ValueError as e:
    print("Validation error:", e)

## 15. `raise` in Data Validation

A common practical use of `raise` is validating data before accepting it.

Example:

In [ ]:
def set_marks(marks):
    if not 0 <= marks <= 100:
        raise ValueError("Marks must be between 0 and 100")

    return marks


print(set_marks(85))

An invalid value such as `150` would raise `ValueError`.

This connects naturally with validation performed by properties and setters.

## 16. `raise` in OOP

Classes can use `raise` to enforce valid object state.

Example:

In [ ]:
class Student:
    def __init__(self, name, marks):
        if not 0 <= marks <= 100:
            raise ValueError("Marks must be between 0 and 100")

        self.name = name
        self.marks = marks


student = Student("Komal", 85)
print(student.name, student.marks)

This works because `85` is within the allowed range.

Creating:

```python
Student("Komal", 150)
```

would raise `ValueError`.

This demonstrates how classes can enforce valid object state during object creation.

## 17. `raise` in a Property Setter

This connects directly with properties and setters.

```python
class Student:
    def __init__(self, marks):
        self._marks = marks

    @property
    def marks(self):
        return self._marks

    @marks.setter
    def marks(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Marks must be between 0 and 100")

        self._marks = value
```

The setter can reject invalid state by raising an exception.

In [ ]:
class Student:
    def __init__(self, marks):
        self._marks = marks

    @property
    def marks(self):
        return self._marks

    @marks.setter
    def marks(self, value):
        if not 0 <= value <= 100:
            raise ValueError("Marks must be between 0 and 100")

        self._marks = value


student = Student(80)

student.marks = 90
print(student.marks)

try:
    student.marks = 150
except ValueError as e:
    print("Error:", e)

The valid assignment succeeds.

The invalid assignment raises `ValueError`, preventing the object from accepting an invalid marks value.

## 18. Raising `TypeError`

Use `TypeError` when the **type** of a value is inappropriate.

Example:

In [ ]:
def calculate_discount(price):
    if not isinstance(price, (int, float)):
        raise TypeError("Price must be a number")

    return price * 0.10


try:
    print(calculate_discount("100"))
except TypeError as e:
    print("Error:", e)

The example connects `raise` with `isinstance()`.

The string `"100"` is not accepted as a numeric price, so the function raises `TypeError`.

## 19. `ValueError` vs `TypeError`

This distinction is important.

### `TypeError`

The type is inappropriate.

For example:

```python
calculate_discount("100")
```

when a number is expected.

### `ValueError`

The type may be correct, but the value is inappropriate.

For example:

```python
calculate_discount(-100)
```

if negative prices are not allowed.

### Mental model

```text
TypeError
→ Wrong type

ValueError
→ Appropriate/general type, invalid value
```

In [ ]:
def calculate_discount(price):
    if not isinstance(price, (int, float)):
        raise TypeError("Price must be a number")

    if price < 0:
        raise ValueError("Price cannot be negative")

    return price * 0.10


for value in [100, "100", -100]:
    try:
        print("Discount:", calculate_discount(value))
    except (TypeError, ValueError) as e:
        print("Error:", e)

The same function uses two different exception types because the two invalid situations are different:

- wrong type → `TypeError`
- invalid numeric value → `ValueError`

## 20. Raising Exceptions in a Function

Here is a more complete validation example:

In [ ]:
def withdraw(balance, amount):

    if not isinstance(amount, (int, float)):
        raise TypeError("Amount must be a number")

    if amount <= 0:
        raise ValueError("Amount must be greater than zero")

    if amount > balance:
        raise ValueError("Insufficient balance")

    return balance - amount


balance = 5000

print(withdraw(balance, 1000))

The function enforces three rules:

1. The amount must be numeric.
2. The amount must be greater than zero.
3. The amount must not exceed the available balance.

Different invalid conditions can therefore raise meaningful exceptions.

## 21. Handling Raised Exceptions at the Caller

A function may raise an exception while the caller decides how to handle it.

Function:

In [ ]:
def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError("Amount must be greater than zero")

    if amount > balance:
        raise ValueError("Insufficient balance")

    return balance - amount

Caller:

In [ ]:
try:
    balance = withdraw(5000, 6000)
    print("Remaining:", balance)

except ValueError as e:
    print("Transaction failed:", e)

This gives an important design pattern:

```text
Function
   ↓
detects invalid condition
   ↓
raises exception
   ↓
Caller
   ↓
decides how to handle it
```

The function does not need to decide how the error should be presented to the user.

## 22. Raising Exceptions in Loops

`raise` can also stop processing when an invalid item is encountered.

In [ ]:
numbers = [10, 20, -5, 30]

try:
    for number in numbers:
        if number < 0:
            raise ValueError("Negative number found")

        print(number)

except ValueError as e:
    print("Error:", e)

The loop prints the valid values before `-5`.

When `-5` is encountered, the `ValueError` is raised and normal loop execution stops.

The surrounding `except` handles the raised exception.

## 23. `raise` with `else` and `finally`

This connects the current topic with the previous notebook.

```python
try:
    age = int(input("Enter age: "))

    if age < 0:
        raise ValueError("Age cannot be negative")

except ValueError as e:
    print("Error:", e)

else:
    print("Valid age")

finally:
    print("Validation completed")
```

The important point is that an explicitly raised exception follows the same `try-except-else-finally` control flow you already learned.

In [ ]:
try:
    age = int(input("Enter age: "))

    if age < 0:
        raise ValueError("Age cannot be negative")

except ValueError as e:
    print("Error:", e)

else:
    print("Valid age")

finally:
    print("Validation completed")

If the age is negative, the `raise` statement sends control to the matching `except`. The `else` block is skipped, while `finally` still executes.

## 24. Re-Raising After Logging

A lower-level function may log or inspect an exception and then re-raise it.

In [ ]:
def process_number(value):
    try:
        number = int(value)
        return 100 / number

    except (ValueError, ZeroDivisionError) as e:
        print("Logging:", e)
        raise


try:
    process_number("abc")
except ValueError:
    print("Caller handled the error")

The lower-level function does not silently hide the problem.

It performs a local action:

```python
print("Logging:", e)
```

and then:

```python
raise
```

propagates the same exception to the caller.

## 25. Common Mistakes

### Mistake 1 — Raising an exception unnecessarily

Do not write:

```python
raise ValueError("Invalid")
```

when there is no actual invalid condition.

Raise an exception because a meaningful program rule has been violated.

### Mistake 2 — Choosing the wrong exception type

Do not use `TypeError` simply because something is invalid.

Consider whether the problem is with the **type** or the **value**.

### Mistake 3 — Using bare `raise` without an active exception

A bare:

```python
raise
```

is used to re-raise an exception currently being handled. It requires an active exception context.

### Mistake 4 — Hiding exceptions

Avoid:

```python
try:
    ...
except:
    pass
```

This silently ignores problems.

### Mistake 5 — Poor exception messages

Prefer:

```python
raise ValueError("Age cannot be negative")
```

over:

```python
raise ValueError("Error")
```

### Mistake 6 — Using exceptions for normal program flow

Exceptions should generally represent exceptional or invalid situations, not ordinary expected branching.

## 26. Summary

By the end of this notebook, you should understand:

- the purpose of `raise`
- raising built-in exceptions
- raising exceptions conditionally
- `raise` with user input
- raising exceptions inside functions
- raising vs returning error values
- meaningful exception messages
- `raise` inside OOP
- `raise` inside property setters
- `TypeError` vs `ValueError`
- handling raised exceptions at the caller
- bare `raise`
- re-raising exceptions
- practical validation patterns

### Next notebook

**`06_Custom_Exceptions.ipynb`**

That notebook will move from built-in exceptions to creating your own exception classes, including inheritance from `Exception`, custom messages, and practical custom exception designs.